# 🏥 MedRAX Reproduction: Multimodal Vision Evaluation on ChestAgentBench
### Reproducing "MedRAX: Medical Reasoning Agent for Chest X-ray" (ICML 2025)
This notebook performs **genuine multimodal reproduction** of the MedRAX benchmark using free Vision-Language Models (VLM).

| Component | Specification |
| :--- | :--- |
| **Benchmark** | **ChestAgentBench** (2,500 expert-curated clinical questions across 7 clinical axes from Eurorad) |
| **Vision Model** | **Google Gemini 1.5 Flash / 2.0 Flash** (Multimodal VLM with native high-res imaging support) |
| **Key Rotation** | **Multi-Key Pool (3 Free Keys)**: 3 keys × 1,500 = **4,500 requests/day** (Covers all 2,500 questions in 1 day!) |
| **Cost & Limits** | **100% Free** via Google AI Studio (No credit card needed) |
| **Agent Tools** | **CheXagent-8B VQA**, TorchXRayVision Classifier & Segmentation, Report Generator (Hosted on Colab GPU) |
| **Storage** | Google Drive persistent cache (batch commits every 10 questions with `os.fsync`) |


## 1. ⚙️ Hardware & GPU Environment Diagnostics
We verify the Colab runtime environment (CUDA availability, GPU model, and VRAM).
The Colab GPU is utilized 100% for hosting MedRAX's specialized local clinical tools (**CheXagent-8B**, **TorchXRayVision**).


In [ ]:
# Check GPU availability and specifications
import os
import sys
import torch

print(f"Python Version : {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Detected GPU   : {device_name} ({vram_gb:.2f} GB VRAM)")
    print("🚀 Local GPU is fully available for MedRAX specialized clinical vision tools!")
else:
    print("⚠️ WARNING: No GPU detected! Please enable GPU in Runtime -> Change runtime type.")

!nvidia-smi


## 2. 💾 Google Drive Mounting & Persistent Storage
Mount Google Drive to persist downloaded datasets, model checkpoints, and evaluation results.
Batch commits ensure that results are flushed directly to Google Drive every 10 questions.


In [ ]:
# Mount Google Drive
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/MedRAX_Thesis')

PERSISTENT_DIRS = {
    "checkpoints": DRIVE_ROOT / "checkpoints",
    "data": DRIVE_ROOT / "data" / "chestagentbench",
    "results": DRIVE_ROOT / "results",
    "logs": DRIVE_ROOT / "logs",
}

for name, path in PERSISTENT_DIRS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"📁 Verified persistent directory: {path}")

# Set environment variables
os.environ["MEDRAX_MODEL_DIR"] = str(PERSISTENT_DIRS["checkpoints"])
os.environ["BENCHMARK_DATA_DIR"] = str(PERSISTENT_DIRS["data"])
os.environ["MEDRAX_RESULTS_DIR"] = str(PERSISTENT_DIRS["results"])
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

print("
✅ Environment variables configured successfully.")


## 3. 🐙 Clone Reproduction Repository from GitHub
Clone the reproduction suite from GitHub. If already present, pull the latest code updates.


In [ ]:
# Configure GitHub repository
import os
import shutil

REPO_URL = "https://github.com/Uyhuynh23/MedRAX-ChestAgentBench.git"
LOCAL_REPO_DIR = Path("/content/MedRAX-ChestAgentBench")

if not LOCAL_REPO_DIR.exists():
    print(f"Cloning {REPO_URL} into {LOCAL_REPO_DIR} ...")
    !git clone {REPO_URL} {LOCAL_REPO_DIR}
else:
    print(f"Repository exists. Pulling latest updates...")
    %cd {LOCAL_REPO_DIR}
    !git pull origin main

%cd {LOCAL_REPO_DIR}
!python scripts/setup_drive.py


## 4. 📦 Install Dependencies & Specialized Radiology Packages
Install PyTorch, LangChain, TorchXRayVision, Transformers, and Google Generative AI SDK.


In [ ]:
%cd /content/MedRAX-ChestAgentBench

# Install project dependencies
!pip install -q -r requirements.txt
!pip install -q torchxrayvision
!pip install -q -U langchain-google-genai google-generativeai

print("
✅ All dependencies installed successfully.")


## 5. 🔑 Configure 3 Free Gemini API Keys (Multi-Key Pool)

To run all 2,500 questions in 1 day without hitting the 1,500 daily quota limit:
1. Create 3 free API keys using 3 different Google accounts at [Google AI Studio](https://aistudio.google.com/apikey).
2. Paste the 3 keys below (either comma-separated or one by one).
3. The built-in **Key Pool** will automatically rotate between keys round-robin to maximize RPM, and instantly failover if any key reaches its daily quota.


In [ ]:
# -------------------------------------------------------------
# Setup 3 Free Gemini API Keys (Comma-separated or Individual)
# -------------------------------------------------------------
import os
from getpass import getpass

# Check Colab Secrets or Environment first
keys = []
try:
    from google.colab import userdata
    for var in ['GEMINI_API_KEY_1', 'GEMINI_API_KEY_2', 'GEMINI_API_KEY_3', 'GOOGLE_API_KEY', 'GEMINI_API_KEY']:
        try:
            val = userdata.get(var)
            if val:
                for k in val.split(','):
                    if k.strip() and k.strip() not in keys:
                        keys.append(k.strip())
        except Exception:
            pass
except Exception:
    pass

# Interactive input if fewer than 3 keys found
if len(keys) < 3:
    print("🔑 Enter your 3 Gemini API keys (from https://aistudio.google.com/apikey):")
    for i in range(len(keys) + 1, 4):
        k = getpass(f"Enter Gemini API Key #{i} (or press Enter if done): ").strip()
        if k and k not in keys:
            keys.append(k)

# Combine into comma-separated pool
pool_str = ",".join(keys)
os.environ["GEMINI_API_KEY"] = pool_str
os.environ["GOOGLE_API_KEY"] = pool_str

print(f"\n✅ Registered {len(keys)} API Key(s) into active Key Pool.")
for idx, k in enumerate(keys, 1):
    masked = f"{k[:6]}...{k[-6:]}" if len(k) > 12 else "***"
    print(f"   Key #{idx}: {masked}")


In [ ]:
# -------------------------------------------------------------
# 🧪 Multi-Key Vision Diagnostic (Smoke Test All 3 Keys)
# -------------------------------------------------------------
# Verifies each key individually with a sample image to ensure all 3 work!
import google.generativeai as genai
from PIL import Image
import numpy as np

test_img = Image.fromarray(np.random.randint(50, 200, (128, 128, 3), dtype=np.uint8))
keys_to_test = [k.strip() for k in os.environ.get("GEMINI_API_KEY", "").split(",") if k.strip()]

print(f"=======================================================")
print(f" Running Multi-Key Smoke Test ({len(keys_to_test)} Keys)")
print(f"=======================================================\n")

all_passed = True
for idx, key in enumerate(keys_to_test, 1):
    masked = f"{key[:6]}...{key[-6:]}" if len(key) > 12 else "***"
    try:
        genai.configure(api_key=key)
        model = genai.GenerativeModel("gemini-1.5-flash")
        resp = model.generate_content([
            "Confirm visual processing capability in 5 words.",
            test_img
        ])
        print(f"  [Key #{idx}] {masked:<20} : ✅ ACTIVE & VISION VERIFIED -> "{resp.text.strip()}"")
    except Exception as e:
        all_passed = False
        print(f"  [Key #{idx}] {masked:<20} : ❌ FAILED -> {e}")

print()
if all_passed and len(keys_to_test) >= 3:
    print(f"🎉 EXCELLENT! All {len(keys_to_test)} keys are fully verified and operational.")
    print(f"🚀 Combined daily capacity: {len(keys_to_test) * 1500:,} requests/day (More than enough for all 2,500 questions!)")
elif all_passed:
    print(f"✅ All {len(keys_to_test)} tested key(s) are active and verified.")
else:
    print("⚠️ One or more keys failed. Please check the error message and ensure the key string is correct.")


## 6. 📥 Download ChestAgentBench Dataset & Extract Figures
Download the **ChestAgentBench** benchmark dataset (`wanglab/chest-agent-bench`) from Hugging Face into Google Drive.
If already present on Google Drive, it will verify integrity and skip re-downloading.


In [ ]:
# Download / Verify ChestAgentBench in persistent Google Drive
%cd /content/MedRAX-ChestAgentBench
!python scripts/download_benchmark.py --data-dir "$BENCHMARK_DATA_DIR"

!ls -lh "$BENCHMARK_DATA_DIR"
!head -n 2 "$BENCHMARK_DATA_DIR/metadata.jsonl" | cut -c 1-120


## 7. 🔍 Interactive Dataset Exploration
Inspect a sample clinical question, options, ground truth answer, and view the chest radiograph.


In [ ]:
import json
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

metadata_file = Path(os.environ["BENCHMARK_DATA_DIR"]) / "metadata.jsonl"
figures_dir = Path(os.environ["BENCHMARK_DATA_DIR"]) / "figures"

with open(metadata_file, "r", encoding="utf-8") as f:
    first_case = json.loads(f.readline())

print("📋 Case ID    :", first_case.get("case_id"))
print("❓ Question ID:", first_case.get("question_id"))
print("🏷️ Category   :", first_case.get("category") or first_case.get("categories"))
print("🎯 Gold Answer:", first_case.get("answer"))
print("\n📝 Question:")
print(first_case.get("question")[:400] + "...")

images = first_case.get("images", [])
if isinstance(images, str):
    images = [images]

if images:
    sample_img_rel = images[0].replace("figures/", "") if images[0].startswith("figures/") else images[0]
    sample_img_path = figures_dir / sample_img_rel
    if sample_img_path.exists():
        img = Image.open(sample_img_path)
        plt.figure(figsize=(5, 5))
        plt.imshow(img, cmap="gray")
        plt.title(f"Case {first_case.get('case_id')} - {sample_img_rel}")
        plt.axis("off")
        plt.show()


## 8. 🧪 Benchmark Execution & Evaluation (Gemini Multimodal)

Now we evaluate on **ChestAgentBench** with Gemini Vision:
1. **Smoke Test**: Fast 2-case verification (~1 minute).
2. **Option A (Baseline Multimodal)**: Direct VLM inference where Gemini reads the radiographs natively and answers the questions.
3. **Option B (MedRAX Agent Mode)**: Gemini acts as the master clinical agent, reading the images AND coordinating specialized tools (**CheXagent-8B VQA**, TorchXRayVision classification & segmentation).


In [ ]:
# -------------------------------------------------------------
# 🧪 Step 8.1: Smoke Test - Verify Pipeline on 2 Cases (~1 min)
# -------------------------------------------------------------
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py     --mode baseline     --model gemini-1.5-flash     --data-dir "$BENCHMARK_DATA_DIR"     --output-dir "$MEDRAX_RESULTS_DIR/smoke_test_gemini"     --max-cases 2     --reset


In [ ]:
# -------------------------------------------------------------
# 📊 Option A: Full Multimodal Baseline Evaluation (All 2,500 Questions)
# -------------------------------------------------------------
# Gemini reads high-resolution radiographs directly with native vision encoder.
# Uses batch commits every 10 questions with fsync directly to Google Drive.
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py     --mode baseline     --model gemini-1.5-flash     --data-dir "$BENCHMARK_DATA_DIR"     --output-dir "$MEDRAX_RESULTS_DIR/gemini_baseline_full" \
    --reset  # Remove --reset if resuming an interrupted session


In [ ]:
# -------------------------------------------------------------
# 🏥 Option B: Full MedRAX Agent Evaluation (All 2,500 Questions)
# -------------------------------------------------------------
# Gemini coordinates local tools (CheXagent-8B, TorchXRayVision) + Vision.
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py     --mode medrax     --model gemini-1.5-flash     --data-dir "$BENCHMARK_DATA_DIR"     --output-dir "$MEDRAX_RESULTS_DIR/gemini_medrax_full" \
    --reset  # Remove --reset if resuming an interrupted session


## 9. 📊 Comparative Analytics: Baseline vs MedRAX Agent
Generate side-by-side comparison tables and accuracy breakdown charts across the 7 clinical axes:
* **Detection**
* **Classification**
* **Localization**
* **Comparison**
* **Relationship**
* **Diagnosis**
* **Characterization**


In [ ]:
# -------------------------------------------------------------
# ⚖️ Compare Baseline vs MedRAX Agent Performance
# -------------------------------------------------------------
%cd /content/MedRAX-ChestAgentBench

!python experiments/compare_runs.py     "$MEDRAX_RESULTS_DIR/gemini_baseline_full/predictions.jsonl"     "$MEDRAX_RESULTS_DIR/gemini_medrax_full/predictions.jsonl"     --output-dir "$MEDRAX_RESULTS_DIR/comparisons"

# Display generated comparison plot
from IPython.display import Image, display
chart_path = Path(os.environ["MEDRAX_RESULTS_DIR"]) / "comparisons" / "accuracy_breakdown.png"
if chart_path.exists():
    display(Image(str(chart_path)))


## 10. 🔄 Synchronize Results & Updates to GitHub
Push any notebook changes or evaluation logs back to your GitHub repository.


In [ ]:
# Configure Git credentials in Colab
!git config --global user.name "Uy Huynh"
!git config --global user.email "huynhtanuy123@gmail.com"

# Check Git status
%cd /content/MedRAX-ChestAgentBench
!git status -s

# Uncomment below to commit & push changes:
# !git add .
# !git commit -m "docs: evaluate multimodal Gemini Vision on ChestAgentBench"
# !git push origin main
